# 🛡️ NIDS-Mamba — Full GPU Training Notebook

**Real-Time Cyber Attack Detection using Mamba (Selective State Space Models)**

## Upload Instructions

### Option 1: Kaggle Notebooks (Recommended — free GPU)
1. Go to [kaggle.com/code](https://www.kaggle.com/code)
2. Click **"+ New Notebook"**
3. File → Upload Notebook → select this `.ipynb` file
4. On the right sidebar: Settings → Accelerator → **GPU T4 x2** (or P100)
5. Run all cells

### Option 2: Google Colab
1. Go to [colab.research.google.com](https://colab.research.google.com)
2. File → Upload Notebook → select this `.ipynb` file
3. Runtime → Change runtime type → **T4 GPU**
4. Run all cells

---
**This notebook is self-contained** — it installs all dependencies, downloads the dataset, preprocesses data, trains the full MambaNIDS model, and evaluates it.

## 1. Environment Setup

In [ ]:
# Install dependencies
!pip install -q mamba-ssm causal-conv1d kagglehub torch pandas numpy scikit-learn matplotlib seaborn

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
import time
import warnings
warnings.filterwarnings('ignore')

# Check GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')
else:
    print('WARNING: No GPU detected! This notebook is designed for GPU training.')

## 2. Download & Preprocess UNSW-NB15

In [ ]:
import kagglehub

# Download dataset
dataset_path = kagglehub.dataset_download('mrwellsdavid/unsw-nb15')
print(f'Dataset path: {dataset_path}')

import os
from pathlib import Path

# Find the CSV files
dataset_path = Path(dataset_path)
csv_files = list(dataset_path.rglob('*.csv'))
print('Found CSV files:')
for f in csv_files:
    print(f'  {f.name} ({f.stat().st_size / 1e6:.1f} MB)')

# Load train and test
train_file = [f for f in csv_files if 'training' in f.name.lower()][0]
test_file = [f for f in csv_files if 'testing' in f.name.lower()][0]

train_df = pd.read_csv(train_file)
test_df = pd.read_csv(test_file)
print(f'\nTrain: {train_df.shape}, Test: {test_df.shape}')

In [ ]:
# Data cleaning
CATEGORICAL_COLS = ['proto', 'service', 'state']

for df in [train_df, test_df]:
    # Drop id
    if 'id' in df.columns:
        df.drop(columns=['id'], inplace=True)
    
    # Standardize attack_cat
    df['attack_cat'] = df['attack_cat'].astype(str).str.strip()
    mask_normal = (df['label'] == 0) | (df['attack_cat'].isin(['', 'nan', 'NaN', ' ']))
    df.loc[mask_normal, 'attack_cat'] = 'Normal'
    df['attack_cat'] = df['attack_cat'].replace({'': 'Normal', 'nan': 'Normal', ' ': 'Normal'})
    
    # Fill missing values
    num_cols = df.select_dtypes(include=[np.number]).columns
    for col in num_cols:
        if df[col].isnull().any():
            df[col].fillna(df[col].median(), inplace=True)
    cat_cols = df.select_dtypes(include=['object']).columns
    for col in cat_cols:
        if df[col].isnull().any():
            df[col].fillna(df[col].mode()[0], inplace=True)

# Encode categoricals
label_encoders = {}
for col in CATEGORICAL_COLS:
    le = LabelEncoder()
    combined = pd.concat([train_df[col].astype(str), test_df[col].astype(str)])
    le.fit(combined)
    train_df[col] = le.transform(train_df[col].astype(str))
    test_df[col] = le.transform(test_df[col].astype(str))
    label_encoders[col] = le

# Encode attack_cat
attack_le = LabelEncoder()
combined_attack = pd.concat([train_df['attack_cat'], test_df['attack_cat']])
attack_le.fit(combined_attack)
train_df['attack_label'] = attack_le.transform(train_df['attack_cat'])
test_df['attack_label'] = attack_le.transform(test_df['attack_cat'])
class_names = list(attack_le.classes_)
num_classes = len(class_names)
print(f'Classes ({num_classes}): {class_names}')

# Scale numeric features
exclude_cols = {'label', 'attack_cat', 'attack_label'}
feature_cols = [c for c in train_df.columns if c not in exclude_cols]
numeric_feature_cols = train_df[feature_cols].select_dtypes(include=[np.number]).columns.tolist()

scaler = StandardScaler()
train_df[numeric_feature_cols] = scaler.fit_transform(train_df[numeric_feature_cols])
test_df[numeric_feature_cols] = scaler.transform(test_df[numeric_feature_cols])

input_dim = len(feature_cols)
print(f'Input dim: {input_dim}')
print(f'\nClass distribution (train):')
print(train_df['attack_cat'].value_counts())

## 3. Sequence Dataset

In [ ]:
class NIDSSequenceDataset(Dataset):
    def __init__(self, df, feature_cols, seq_len=32, stride=1):
        features = df[feature_cols].values.astype(np.float32)
        labels = df['attack_label'].values.astype(np.int64)
        self.indices = list(range(0, len(features) - seq_len + 1, stride))
        self.features = features
        self.labels = labels
        self.seq_len = seq_len
    
    def __len__(self):
        return len(self.indices)
    
    def __getitem__(self, idx):
        start = self.indices[idx]
        end = start + self.seq_len
        x = torch.tensor(self.features[start:end], dtype=torch.float32)
        y = torch.tensor(self.labels[end - 1], dtype=torch.long)
        return x, y

SEQ_LEN = 32
BATCH_SIZE = 128

train_dataset = NIDSSequenceDataset(train_df, feature_cols, seq_len=SEQ_LEN)
test_dataset = NIDSSequenceDataset(test_df, feature_cols, seq_len=SEQ_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, drop_last=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f'Train sequences: {len(train_dataset):,}')
print(f'Test sequences:  {len(test_dataset):,}')

x_sample, y_sample = next(iter(train_loader))
print(f'Batch shapes: X={x_sample.shape}, Y={y_sample.shape}')

## 4. MambaNIDS Model (using mamba-ssm)

In [ ]:
from mamba_ssm import Mamba

class MambaNIDS(nn.Module):
    """
    Mamba-based NIDS classifier using the official mamba-ssm library.
    """
    def __init__(self, input_dim, num_classes=10, d_model=64,
                 n_layers=4, d_state=16, d_conv=4, expand=2, dropout=0.1):
        super().__init__()
        self.input_proj = nn.Linear(input_dim, d_model)
        self.input_norm = nn.LayerNorm(d_model)
        
        self.layers = nn.ModuleList()
        self.norms = nn.ModuleList()
        for _ in range(n_layers):
            self.layers.append(
                Mamba(
                    d_model=d_model,
                    d_state=d_state,
                    d_conv=d_conv,
                    expand=expand,
                )
            )
            self.norms.append(nn.LayerNorm(d_model))
        
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, num_classes),
        )
    
    def forward(self, x):
        x = self.input_proj(x)
        x = self.input_norm(x)
        for layer, norm in zip(self.layers, self.norms):
            residual = x
            x = norm(layer(x) + residual)
        x = x.mean(dim=1)  # global average pool
        x = self.dropout(x)
        return self.classifier(x)

# Build model
model = MambaNIDS(
    input_dim=input_dim,
    num_classes=num_classes,
    d_model=64,
    d_state=16,
    n_layers=4,
    expand=2,
).to(device)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model parameters: {n_params:,}')
print(model)

## 5. Training

In [ ]:
N_EPOCHS = 15
LR = 1e-3

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=N_EPOCHS)

train_losses = []
train_accs = []
val_accs = []

best_val_acc = 0.0
total_start = time.time()

for epoch in range(1, N_EPOCHS + 1):
    # Train
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for batch_idx, (x_batch, y_batch) in enumerate(train_loader):
        x_batch, y_batch = x_batch.to(device), y_batch.to(device)
        
        optimizer.zero_grad()
        logits = model(x_batch)
        loss = criterion(logits, y_batch)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        
        running_loss += loss.item()
        _, predicted = logits.max(1)
        total += y_batch.size(0)
        correct += predicted.eq(y_batch).sum().item()
    
    scheduler.step()
    train_loss = running_loss / len(train_loader)
    train_acc = correct / total
    train_losses.append(train_loss)
    train_accs.append(train_acc)
    
    # Validate
    model.eval()
    val_correct = 0
    val_total = 0
    with torch.no_grad():
        for x_batch, y_batch in test_loader:
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)
            logits = model(x_batch)
            _, predicted = logits.max(1)
            val_total += y_batch.size(0)
            val_correct += predicted.eq(y_batch).sum().item()
    
    val_acc = val_correct / val_total
    val_accs.append(val_acc)
    
    print(f'Epoch {epoch:2d}/{N_EPOCHS}: loss={train_loss:.4f}, '
          f'train_acc={train_acc:.4f}, val_acc={val_acc:.4f}')
    
    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save({
            'model_state_dict': model.state_dict(),
            'epoch': epoch,
            'val_acc': val_acc,
            'config': {
                'input_dim': input_dim, 'num_classes': num_classes,
                'd_model': 64, 'n_layers': 4, 'd_state': 16, 'expand': 2,
            }
        }, 'mamba_nids_best.pt')
        print(f'  → Saved best model (val_acc={val_acc:.4f})')

total_time = time.time() - total_start
print(f'\nTraining complete in {total_time:.1f}s')
print(f'Best validation accuracy: {best_val_acc:.4f}')

## 6. Training Curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

epochs = range(1, len(train_losses) + 1)

ax1.plot(epochs, train_losses, 'b-o', label='Train Loss')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training Loss')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(epochs, train_accs, 'g-o', label='Train Acc')
ax2.plot(epochs, val_accs, 'r-s', label='Val Acc')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.set_title('Accuracy')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('mamba_training_curves.png', dpi=150)
plt.show()

## 7. Final Evaluation

In [ ]:
# Load best model
checkpoint = torch.load('mamba_nids_best.pt')
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for x_batch, y_batch in test_loader:
        x_batch = x_batch.to(device)
        logits = model(x_batch)
        preds = logits.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(y_batch.numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

acc = accuracy_score(all_labels, all_preds)
f1_macro = f1_score(all_labels, all_preds, average='macro', zero_division=0)
f1_weighted = f1_score(all_labels, all_preds, average='weighted', zero_division=0)

print(f'Accuracy:    {acc:.4f}')
print(f'Macro F1:    {f1_macro:.4f}')
print(f'Weighted F1: {f1_weighted:.4f}')
print()
print(classification_report(all_labels, all_preds, target_names=class_names, zero_division=0))

In [ ]:
# Confusion matrix
cm = confusion_matrix(all_labels, all_preds)
fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names, ax=ax)
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title('Confusion Matrix — MambaNIDS (GPU)')
plt.tight_layout()
plt.savefig('mamba_confusion_matrix.png', dpi=150)
plt.show()

In [ ]:
# Inference time benchmark
model.eval()
x_dummy = torch.randn(1, SEQ_LEN, input_dim).to(device)

# Warmup
with torch.no_grad():
    for _ in range(20):
        model(x_dummy)

# Benchmark
if torch.cuda.is_available():
    torch.cuda.synchronize()

times = []
with torch.no_grad():
    for _ in range(100):
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        start = time.perf_counter()
        model(x_dummy)
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        times.append(time.perf_counter() - start)

avg_us = np.mean(times) * 1e6
print(f'Inference time: {avg_us:.2f} µs/sample')

In [ ]:
# Save final checkpoint
torch.save({
    'model_state_dict': model.state_dict(),
    'accuracy': acc,
    'f1_macro': f1_macro,
    'f1_weighted': f1_weighted,
    'class_names': class_names,
    'config': {
        'input_dim': input_dim, 'num_classes': num_classes,
        'd_model': 64, 'n_layers': 4, 'd_state': 16, 'expand': 2,
    },
    'inference_time_us': avg_us,
}, 'mamba_nids_final.pt')

print('\n✅ Training complete! Checkpoint saved as mamba_nids_final.pt')
print('Download this file and place it in model/checkpoints/ in your local repo.')